# Learning 02: RAG Pipeline with LangChain

Retrieval-Augmented Generation: embed documents into a vector store, retrieve relevant ones at query time, and generate a grounded answer.

## Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

OPENAI_API_KEY = "your-api-key-here"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
print("✓ LLM initialized")


In [ ]:
import json, os

# In Jupyter, os.getcwd() returns the notebook's directory (learning/, tasks/, solutions/)
# Fixtures are one level up at ../fixtures/input/
fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))

with open(os.path.join(fixtures, "tickets.json")) as f:
    tickets = json.load(f)

with open(os.path.join(fixtures, "knowledge_base.json")) as f:
    knowledge_base = json.load(f)

with open(os.path.join(fixtures, "test_questions.json")) as f:
    test_questions = json.load(f)

print(f"✓ Loaded {len(tickets)} tickets, {len(knowledge_base)} KB articles, {len(test_questions)} test questions")


## 1. Build the Vector Store

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Extract texts and metadata from the knowledge base
texts = [article["content"] for article in knowledge_base]
metadatas = [{"id": article["id"], "title": article["title"]} for article in knowledge_base]

# Embed and index
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)
vectorstore = FAISS.from_texts(texts, embeddings, metadatas=metadatas)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✓ Indexed {len(texts)} articles")


## 2. Test the Retriever

In [ ]:
docs = retriever.invoke("How long does a refund take?")
for i, doc in enumerate(docs):
    print(f"[{i+1}] {doc.metadata['title']}")
    print(f"    {doc.page_content[:120]}...")
    print()


## 3. Build the RAG Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f"[{i+1}] {doc.page_content}" for i, doc in enumerate(docs)
    )

rag_prompt = ChatPromptTemplate.from_template("""\
Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}

Answer:""")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("How long does a refund take?")
print(answer)


## 4. Evaluate Against Test Questions

In [ ]:
print("RAG Evaluation\n" + "="*50)
for q in test_questions:
    answer = rag_chain.invoke(q["question"])
    hits = [kw for kw in q["expected_keywords"] if kw.lower() in answer.lower()]
    score = len(hits) / len(q["expected_keywords"])
    status = "✓" if score >= 0.5 else "✗"
    print(f"{status} Q{q['id']}: {q['question'][:50]!r}")
    print(f"   Score: {score:.0%} | Found: {hits}")
    print()


## 5. Persist the Index (Optional)

In [ ]:
# Save to disk — avoids re-embedding on next run
vectorstore.save_local("faiss_index")
print("✓ Saved to faiss_index/")

# Load on next run:
# vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
# retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


## Summary

- `FAISS.from_texts()` embeds and indexes documents
- `as_retriever()` turns the vectorstore into an LCEL-compatible Runnable
- RAG chain: `{context: retriever | format_docs, question: RunnablePassthrough()} | prompt | llm | parser`
- Evaluate with expected keywords per question